# 07 — Failure cases on the real-world typographic sets (Colab, CPU is enough)
Which real-world images does the defense repair, which does it break, and which stay wrong for both the reproduced DP and option B?
Uses the cached features in `dp_project/cache_test` (no re-encoding) and the trained tokens in `dp_project/runs.zip`.
Raw images are only needed to draw the example grids.

**Before running:** upload `runs.zip` (the one with `runs/dp_seed0..2`, `runs/vocab_seed0..2`) to `MyDrive/dp_project/`.

In [ ]:
# A) Drive and paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, subprocess
DRIVE = "/content/drive/MyDrive/dp_project"
CACHE = f"{DRIVE}/cache_test"
OUT = f"{DRIVE}/failure_cases"
BASE = "/content"
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(CACHE)))
print("runs.zip present:", os.path.exists(f"{DRIVE}/runs.zip"))

In [ ]:
# B) Authors' code pinned to 50d7887, same two compatibility patches as every notebook
!pip -q install ftfy regex git+https://github.com/openai/CLIP.git
import pathlib
os.chdir(BASE)
if not os.path.isdir("Defense-Prefix"):
    subprocess.run("git clone -q https://github.com/azuma164/Defense-Prefix.git", shell=True, check=True)
os.chdir(f"{BASE}/Defense-Prefix")
subprocess.run("git checkout -q 50d7887", shell=True, check=True)
p = pathlib.Path("datasets/utils/make_dataset_train.py"); s = p.read_text()
if "_textsize" not in s:
    helper = "\ndef _textsize(draw, text, font):\n    # Replacement for ImageDraw.textsize (removed in Pillow 10).\n    l, t, r, b = draw.multiline_textbbox((0, 0), text, font=font)\n    return r, b\n"
    s = s.replace("random.seed(300)", "random.seed(300)\n" + helper, 1).replace("imagedraw.textsize(text, font=font)", "_textsize(imagedraw, text, font)")
    p.write_text(s)
print(subprocess.run("git rev-parse HEAD && git diff --stat", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%writefile /content/Defense-Prefix/cache_features.py
"""Cache CLIP ViT-B/32 image features for one evaluation dataset of the Defense-Prefix paper.

Run from the root of the Defense-Prefix repository, one dataset per process
(the authors' attack generator seeds `random` once at import, and the authors
ran eval.py once per dataset, so one process per dataset reproduces their setup).

The dataset objects are the authors' own classes from `datasets/`, built with the
same arguments as the authors' `dataset.py`, so splits, labels, class names,
prompt templates and synthetic attacks are exactly theirs. The only difference
from `dataset.py` is that `download` is configurable (Stanford Cars can no longer
be downloaded through torchvision).

Features are computed exactly as in the authors' eval.py: fp16 CLIP image
features, L2-normalised. Evaluating any Defense-Prefix token on these cached
features gives the same predictions as eval.py.
"""
import argparse
import json
import os
import subprocess
import sys
import time

import clip
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.insert(0, os.getcwd())

REAL_WORLD = {"rta-100", "paint", "disentangling"}
device = "cuda" if torch.cuda.is_available() else "cpu"


def build(name, preprocess, root="datasets", download=True):
    """Mirror of the authors' dataset.py with a configurable download flag."""
    if name == "imagenet":
        from datasets.ImageNetV2 import ImageNetValDataset
        return ImageNetValDataset(location=root, transform=preprocess)
    if name == "caltech":
        from datasets.caltech101 import Caltech101
        return Caltech101(root=root, transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "pets":
        from datasets.oxford_pets import OxfordIIITPet
        return OxfordIIITPet(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "cars":
        from datasets.stanford_cars import StanfordCars
        return StanfordCars(root=root, split="test", transform=preprocess, download=False, make_typographic_dataset=True)
    if name == "flowers":
        from datasets.flowers102 import Flowers102
        return Flowers102(root=root, split="test", transform=preprocess, download=download, make_typographic_dataset=True)
    if name == "food":
        from datasets.food101 import Food101
        return Food101(root=root, split="test", transform=preprocess, download=download)
    if name == "aircraft":
        from datasets.fgvcaircraft import FGVCAircraft
        return FGVCAircraft(root=root, split="test", transform=preprocess, download=download)
    if name == "dtd":
        from datasets.dtd import DTD
        return DTD(root=root, split="test", transform=preprocess, download=download)
    if name == "eurosat":
        from datasets.eurosat import EuroSAT
        return EuroSAT(root=root, split="test", transform=preprocess, download=download)
    if name == "sun":
        from datasets.sun397 import SUN397
        return SUN397(root=root, split="test", transform=preprocess, download=False)
    if name == "paint":
        from datasets.paint_ta import PAINTDataset
        return PAINTDataset(root=root, transform=preprocess)
    if name == "disentangling":
        from datasets.disentangling_ta import DisentanglingDataset
        return DisentanglingDataset(root=root, transform=preprocess)
    if name == "rta-100":
        from datasets.rta100 import RTA100
        return RTA100(root=root, transform=preprocess)
    raise ValueError(name)


def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out", required=True, help="folder for <dataset>.pt")
    ap.add_argument("--root", default="datasets")
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    model, preprocess = clip.load("ViT-B/32", device=device)
    model.eval()

    t0 = time.time()
    data = build(args.dataset, preprocess, root=args.root)  # also generates the synthetic attack images (seed 300)
    prep_s = time.time() - t0

    loader = DataLoader(data, batch_size=args.batch_size, shuffle=False, num_workers=args.workers)
    real = args.dataset in REAL_WORLD
    clean, attack, labels = [], [], []
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    with torch.no_grad():
        for original_image, typographic_image, target in tqdm(loader, desc=args.dataset):
            f = model.encode_image(original_image.to(device))
            f /= f.norm(dim=-1, keepdim=True)          # identical to eval.py
            clean.append(f.cpu())
            if not real:
                g = model.encode_image(typographic_image.to(device))
                g /= g.norm(dim=-1, keepdim=True)
                attack.append(g.cpu())
            labels.append(torch.as_tensor(target))
    enc_s = time.time() - t1

    out = {
        "dataset": args.dataset,
        "real_world": real,
        "classes": list(data.classes),
        "template": data.templates[0],
        "labels": torch.cat(labels).long(),
        "clean": torch.cat(clean),                       # for real-world sets these images contain the attack
        "attack": torch.cat(attack) if attack else None,
        "meta": {
            "n_images": len(data),
            "n_classes": len(data.classes),
            "prepare_seconds": round(prep_s, 1),
            "encode_seconds": round(enc_s, 1),
            "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if device == "cuda" else None,
            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
            "torch": torch.__version__,
            "repo_commit": git_commit(),
            "model": "ViT-B/32",
        },
    }
    path = os.path.join(args.out, f"{args.dataset}.pt")
    torch.save(out, path)
    print(json.dumps({"saved": path, **out["meta"]}, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/Defense-Prefix/eval_cached.py
"""Evaluate CLIP and CLIP + Defense-Prefix on cached image features.

Text features and scoring follow the authors' eval.py exactly:
one template per dataset (templates[0]), "* {class}" with the * embedding
replaced by the DP vector, cosine similarity x 100, argmax.

Usage (from the Defense-Prefix repo root):
  python eval_cached.py --cache CACHE_DIR --token learned_token/dp_vit-b32.pt --tag authors_token
"""
import argparse
import csv
import glob
import json
import os
import sys

import clip
import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
from utils.non_nv import encode_text_with_learnt_tokens  # authors' function

device = "cuda" if torch.cuda.is_available() else "cpu"

ORDER = ["imagenet", "caltech", "pets", "cars", "flowers", "food", "aircraft", "dtd", "sun", "eurosat",
         "disentangling", "paint", "rta-100"]
# Paper, Tables B, C and 3 (ViT-B/32): (CLIP, CLIP + DP)
PAPER = {
    "clean": {"imagenet": (62.02, 62.48), "caltech": (88.64, 89.28), "pets": (87.35, 87.22), "cars": (58.72, 57.47),
              "flowers": (66.32, 63.82), "food": (84.14, 83.65), "aircraft": (18.99, 19.26), "dtd": (44.57, 40.64),
              "sun": (61.74, 61.41), "eurosat": (42.98, 43.85)},
    "synthetic": {"imagenet": (39.10, 49.83), "caltech": (63.97, 79.54), "pets": (58.95, 72.88), "cars": (21.02, 28.64),
                  "flowers": (31.32, 44.12), "food": (56.27, 67.79), "aircraft": (10.83, 14.49), "dtd": (25.53, 31.6),
                  "sun": (34.02, 43.50), "eurosat": (4.86, 9.65)},
    "real": {"disentangling": (43.27, 71.93), "paint": (50.00, 63.64), "rta-100": (47.20, 58.00)},
}


def load_model():
    model, _ = clip.load("ViT-B/32", device=device)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    model.eval()
    return model


@torch.no_grad()
def class_text_features(model, classes, template, token=None):
    if token is None:
        toks = torch.cat([clip.tokenize(template.format(c)) for c in classes]).to(device)
        f = model.encode_text(toks)
        return f / f.norm(dim=-1, keepdim=True)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in classes]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    f = model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False)
    return F.normalize(f, dim=-1)


@torch.no_grad()
def predictions(img_feats, text_feats, bs=8192):
    out = []
    for i in range(0, len(img_feats), bs):
        sim = 100.0 * (img_feats[i:i + bs].to(device) @ text_feats.T)
        out.append(sim.argmax(-1).cpu())
    return torch.cat(out)


def evaluate(model, cache_dir, token):
    """Returns rows: dataset, condition, n, acc_clip, acc_dp, plus per-image correctness for later stats."""
    rows, per_image = [], {}
    files = {os.path.basename(p)[:-3]: p for p in glob.glob(os.path.join(cache_dir, "*.pt"))}
    for name in [d for d in ORDER if d in files]:
        c = torch.load(files[name], map_location="cpu", weights_only=False)
        t_clip = class_text_features(model, c["classes"], c["template"])
        t_dp = class_text_features(model, c["classes"], c["template"], token)
        conds = [("real", c["clean"])] if c["real_world"] else [("clean", c["clean"]), ("synthetic", c["attack"])]
        for cond, feats in conds:
            y = c["labels"]
            ok_clip = predictions(feats, t_clip).eq(y)
            ok_dp = predictions(feats, t_dp).eq(y)
            per_image[f"{name}/{cond}"] = {"clip": ok_clip, "dp": ok_dp}
            rows.append({"dataset": name, "condition": cond, "n": len(y),
                         "acc_clip": round(100 * ok_clip.float().mean().item(), 2),
                         "acc_dp": round(100 * ok_dp.float().mean().item(), 2)})
    return rows, per_image


def with_paper(rows):
    for r in rows:
        p = PAPER.get(r["condition"], {}).get(r["dataset"])
        r["paper_clip"], r["paper_dp"] = (p if p else (None, None))
    return rows


def print_table(rows):
    print(f"{'dataset':<14}{'cond':<10}{'n':>7} | {'paper CLIP':>10}{'ours CLIP':>10} | {'paper DP':>9}{'ours DP':>9}")
    for r in rows:
        pc = "" if r["paper_clip"] is None else f"{r['paper_clip']:.2f}"
        pd = "" if r["paper_dp"] is None else f"{r['paper_dp']:.2f}"
        print(f"{r['dataset']:<14}{r['condition']:<10}{r['n']:>7} | {pc:>10}{r['acc_clip']:>10.2f} | {pd:>9}{r['acc_dp']:>9.2f}")
    for cond in ["clean", "synthetic", "real"]:
        sub = [r for r in rows if r["condition"] == cond]
        if sub:
            mc = sum(r["acc_clip"] for r in sub) / len(sub)
            md = sum(r["acc_dp"] for r in sub) / len(sub)
            print(f"  mean {cond:<9} over {len(sub)} sets: CLIP {mc:.2f} | DP {md:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--tag", default="authors_token")
    ap.add_argument("--out", default=None, help="results folder (default: <cache>/../results)")
    args = ap.parse_args()

    token = torch.load(args.token, map_location=device, weights_only=False)
    token = token.detach()
    if token.dim() == 1:          # authors save shape (1, 512); eval.py passes token.unsqueeze(0)
        token = token.unsqueeze(0)
    model = load_model()
    token = token.to(device).type(model.dtype)

    rows, per_image = evaluate(model, args.cache, token)
    rows = with_paper(rows)
    print_table(rows)

    out = args.out or os.path.join(os.path.dirname(os.path.abspath(args.cache)), "results")
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, f"{args.tag}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    torch.save(per_image, os.path.join(out, f"{args.tag}_per_image.pt"))
    print("saved", os.path.join(out, f"{args.tag}.csv"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/Defense-Prefix/failure_cases.py
"""Failure-case analysis on the real-world typographic sets (RTA-100, PAINT, optionally Disentangling).

Uses the cached image features (same order as the authors' dataset objects, shuffle=False) to get the
predictions of CLIP, the authors' token, the reproduced DP tokens and the option-B tokens, then pulls the
corresponding raw images from the authors' dataset classes to draw labelled example grids.

Categories (per image):
  fixed       CLIP wrong, every DP seed right           -> what the defense repairs
  broken      CLIP right, every DP seed wrong           -> what the defense breaks
  persistent  every DP seed AND every B seed wrong      -> residual failures neither version fixes
  b_wins      every B seed right, every DP seed wrong
  b_loses     every DP seed right, every B seed wrong

Image order: the authors' real-world loaders list files with os.listdir(), whose order depends on the file system,
so the order in which the features were cached (Colab) need not match the order on the machine running this script.
Each cached feature is therefore matched to its image by re-encoding the images and taking the nearest neighbour
(cosine ~ 1); the match is checked to be one-to-one and label-consistent.

Runs on CPU (re-encoding 1,000 images takes a few minutes; a GPU is used if present). Run from the Defense-Prefix repo root:
  python failure_cases.py --cache CACHE_DIR --runs RUNS_DIR --sets rta-100 paint --out OUT_DIR
"""
import argparse
import csv
import json
import os
import random
import sys

import torch

sys.path.insert(0, os.getcwd())
from cache_features import build              # authors' dataset classes, same arguments as feature caching
from eval_cached import class_text_features, load_model, device

DP = ["dp_seed0", "dp_seed1", "dp_seed2"]
B = ["vocab_seed0", "vocab_seed1", "vocab_seed2"]
CATS = ["fixed", "broken", "persistent", "b_wins", "b_loses"]
# "Soft Nordic" pastel palette, same as scripts/make_figures.py
PASTEL = {"gray": "#cfcbc4", "blue": "#6b8fc2", "apricot": "#efab7d", "mint": "#9fcfbf"}
CAT_COLOR = {"fixed": PASTEL["mint"], "broken": PASTEL["apricot"], "persistent": PASTEL["gray"],
             "b_wins": PASTEL["blue"], "b_loses": PASTEL["apricot"]}
CAT_LABEL = {"fixed": "Fixed by DP", "broken": "Broken by DP", "persistent": "Wrong for DP and DP + everyday words",
             "b_wins": "Only DP + everyday words right", "b_loses": "Only DP right"}
CAT_SHORT = {"fixed": "Fixed\nby DP", "broken": "Broken\nby DP", "persistent": "Wrong for both\n(DP, DP + everyday)",
             "b_wins": "Only DP +\neveryday right", "b_loses": "Only DP\nright"}
INK, INK2 = "#0b0b0b", "#52514e"


def edge(c, f=0.72):
    c = c.lstrip("#"); return "#" + "".join(f"{int(int(c[i:i + 2], 16) * f):02x}" for i in (0, 2, 4))


def load_token(path, dtype):
    t = torch.load(path, map_location=device, weights_only=False).detach()
    return (t.unsqueeze(0) if t.dim() == 1 else t).to(device).type(dtype)


def match_images(name, model, c, root):
    """Map every cached feature to its image file (see module docstring). Returns (dataset, list of file indices)."""
    from clip.clip import _transform
    from torch.utils.data import DataLoader
    data = build(name, _transform(224), root=root)
    feats = []
    with torch.no_grad():
        for img, _, _ in DataLoader(data, batch_size=64, shuffle=False, num_workers=2):
            f = model.encode_image(img.to(device).type(model.dtype)).float()
            feats.append(torch.nn.functional.normalize(f, dim=-1).cpu())
    feats = torch.cat(feats)
    sim = c["clean"].float() @ feats.T
    best, j = sim.max(dim=1)
    assert len(set(j.tolist())) == len(j), f"{name}: image matching is not one-to-one"
    bad = sum(data.true_labels[int(k)] != c["classes"][int(l)] for k, l in zip(j, c["labels"]))
    assert bad == 0, f"{name}: {bad} matched images have a different label"
    print(f"{name}: matched {len(j)} cached features to images, cosine min {best.min():.4f} / mean {best.mean():.4f}")
    return data, j.tolist()


def written_text(fname):
    """'label=cup_text=iPhone.jpg' -> 'iPhone' (RTA-100 and PAINT file-name convention)."""
    try:
        return fname.split("_text=")[1].rsplit(".", 1)[0]
    except IndexError:
        return None


def majority(preds):
    return max(set(preds), key=preds.count)


def grid(items, classes, title, path, color, ncol=4):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if not items:
        return
    nrow = (len(items) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.9 * nrow + 0.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (img, r) in zip(axes.ravel(), items):
        ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
        ax.imshow(img)
        for sp in ax.spines.values():          # pastel frame = category colour
            sp.set_color(color); sp.set_linewidth(5)
        ax.set_title(f"true: {classes[r['true']]}   written: {r['text']}\nCLIP: {classes[r['clip']]}\n"
                     f"DP: {classes[r['dp_major']]}\nDP + everyday: {classes[r['b_major']]}",
                     fontsize=7.5, loc="left", color=INK2)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / (3.9 * nrow + 0.6)), h_pad=2.2)
    fig.suptitle(title, fontsize=11, x=0.01, y=0.995, ha="left", va="top", color=INK,
                 bbox={"boxstyle": "round,pad=0.35", "facecolor": color, "edgecolor": edge(color)})
    fig.savefig(path, dpi=150, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)


def counts_figure(summary, path):
    """Share of images per category for each real-world set (pastel bars, one colour per set)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    sets = list(summary)
    cols = [PASTEL["blue"], PASTEL["apricot"], PASTEL["mint"]]
    fig, ax = plt.subplots(figsize=(7.4, 3.0))
    w = 0.8 / len(sets)
    for i, name in enumerate(sets):
        n = summary[name]["counts"]["n"]
        vals = [100 * summary[name]["counts"][c] / n for c in CATS]
        x = np.arange(len(CATS)) + (i - (len(sets) - 1) / 2) * w
        ax.bar(x, vals, w * 0.92, color=cols[i % 3], edgecolor=edge(cols[i % 3]), linewidth=0.7,
               hatch="///" if i == 2 else None, label=f"{name} (n={n})")
        for xi, v, c in zip(x, vals, CATS):
            ax.text(xi, v + 0.6, f"{v:.1f}", ha="center", va="bottom", fontsize=6.5, color=INK2)
    ax.set_xticks(range(len(CATS)))
    ax.set_xticklabels([CAT_SHORT[c] for c in CATS], fontsize=7.5)
    ax.set_ylabel("% of images", color=INK2)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.grid(axis="y", color="#e4e3df"); ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--runs", required=True)
    ap.add_argument("--authors_token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--sets", nargs="+", default=["rta-100", "paint"])
    ap.add_argument("--root", default="datasets")
    ap.add_argument("--out", required=True)
    ap.add_argument("--per_cat", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    import torchvision.transforms as T
    show = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)])
    model = load_model()
    toks = {"authors": load_token(args.authors_token, model.dtype)}
    toks.update({t: load_token(f"{args.runs}/{t}/dp_token.pt", model.dtype) for t in DP + B})
    summary = {}
    for name in args.sets:
        c = torch.load(f"{args.cache}/{name}.pt", map_location="cpu", weights_only=False)
        x = c["clean"].float().to(device)
        preds = {"clip": (x @ class_text_features(model, c["classes"], c["template"]).float().T).argmax(-1).cpu()}
        for k, t in toks.items():
            preds[k] = (x @ class_text_features(model, c["classes"], c["template"], t).float().T).argmax(-1).cpu()
        y = c["labels"]
        data, file_idx = match_images(name, model, c, args.root)
        acc = {k: round(100 * p.eq(y).float().mean().item(), 2) for k, p in preds.items()}
        print(name, "accuracy check (compare with results/*.csv):", acc)

        rows = []
        for i in range(len(y)):
            fname = data.img_files[file_idx[i]]
            r = {"idx": i, "file": fname, "text": written_text(fname), "true": int(y[i]),
                 **{k: int(p[i]) for k, p in preds.items()}}
            dp_ok = [r[k] == r["true"] for k in DP]
            b_ok = [r[k] == r["true"] for k in B]
            clip_ok = r["clip"] == r["true"]
            r["dp_major"], r["b_major"] = majority([r[k] for k in DP]), majority([r[k] for k in B])
            r["fixed"] = (not clip_ok) and all(dp_ok)
            r["broken"] = clip_ok and not any(dp_ok)
            r["persistent"] = not any(dp_ok) and not any(b_ok)
            r["b_wins"] = all(b_ok) and not any(dp_ok)
            r["b_loses"] = all(dp_ok) and not any(b_ok)
            # residual failure that lands on CLIP's own (attacked) answer: the defense did not move the prediction
            r["persistent_same_as_clip"] = r["persistent"] and r["dp_major"] == r["clip"]
            r.update({"true_name": c["classes"][r["true"]], "clip_name": c["classes"][r["clip"]],
                      "dp_name": c["classes"][r["dp_major"]], "b_name": c["classes"][r["b_major"]]})
            # did the model output the WRITTEN word (attack success) rather than some other wrong class?
            r["clip_is_text"] = r["clip_name"] == r["text"]
            r["dp_is_text"] = r["dp_name"] == r["text"]
            r["b_is_text"] = r["b_name"] == r["text"]
            rows.append(r)
        with open(f"{args.out}/preds_{name}.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        counts = {k: sum(r[k] for r in rows) for k in CATS + ["persistent_same_as_clip"]}
        counts["n"] = len(rows)
        pers = [r for r in rows if r["persistent"]]
        text_stats = {"attack_success_clip": sum(r["clip_is_text"] for r in rows),
                      "attack_success_dp_majority": sum(r["dp_is_text"] for r in rows),
                      "attack_success_everyday_majority": sum(r["b_is_text"] for r in rows),
                      "persistent_dp_predicts_text": sum(r["dp_is_text"] for r in pers),
                      "persistent_clip_predicts_text": sum(r["clip_is_text"] for r in pers)}
        summary[name] = {"accuracy": acc, "counts": counts, "written_word": text_stats}
        print(name, text_stats)
        print(name, counts)

        from PIL import Image
        rng = random.Random(args.seed)
        for cat in CATS:
            pick = [r for r in rows if r[cat]]
            rng.shuffle(pick)
            items = [(show(Image.open(os.path.join(data.img_dir, r["file"])).convert("RGB")), r) for r in pick[:args.per_cat]]
            grid(items, c["classes"], f"{name}: {CAT_LABEL[cat]} ({counts[cat]} of {len(rows)} images)",
                 f"{args.out}/{name}_{cat}.png", CAT_COLOR[cat])
    json.dump(summary, open(f"{args.out}/failure_summary.json", "w"), indent=1)
    counts_figure(summary, f"{args.out}/fig6_failure_categories.png")
    print("saved", args.out)


if __name__ == "__main__":
    main()


In [ ]:
# C) Raw images of the real-world sets (same layout as notebook 02-1)
os.chdir(f"{BASE}/Defense-Prefix")
if not os.path.isdir("datasets/ours/data"):
    !unzip -q -o rta100.zip -d /tmp/rta && mkdir -p datasets/ours && rm -rf datasets/ours/data && mv /tmp/rta/rta100 datasets/ours/data
if not os.path.isdir("datasets/typographic_real_data_from_PAINT/typographic_attacks"):
    !rm -rf /tmp/patching && git clone -q --depth 1 https://github.com/mlfoundations/patching.git /tmp/patching
    !mkdir -p datasets/typographic_real_data_from_PAINT && unzip -q -o /tmp/patching/typographic_attack_data.zip -d datasets/typographic_real_data_from_PAINT
SETS = ["rta-100", "paint"]
# Disentangling only if the rebuilt layout from notebook 02-1 was kept in Drive (annotations.json, classes.json, images/)
src = f"{DRIVE}/typographic_attack_dataset"
if os.path.isdir(src) and not os.path.isdir("datasets/typographic_attack_dataset"):
    !cp -r "{src}" datasets/typographic_attack_dataset
if os.path.isdir("datasets/typographic_attack_dataset"):
    SETS.append("disentangling")
print("sets:", SETS)

In [ ]:
# D) Trained tokens
if not os.path.isdir("/content/runs"):
    !unzip -q -o "{DRIVE}/runs.zip" -d /content
print(sorted(os.listdir("/content/runs")))

In [ ]:
# E) Run (CPU is fine; ~1-2 min). The printed accuracies must match results/*.csv
os.chdir(f"{BASE}/Defense-Prefix")
SETS_STR = " ".join(SETS)
!python failure_cases.py --cache {CACHE} --runs /content/runs --sets {SETS_STR} --out {OUT}
print(open(f"{OUT}/failure_summary.json").read())

In [ ]:
# F) Look at the grids
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT}/*.png")):
    print(os.path.basename(f)); display(Image(f, width=900))

In [ ]:
# G) Pack for download (also stays in Drive: dp_project/failure_cases)
!cd {DRIVE} && zip -q -r failure_cases.zip failure_cases && ls -la {DRIVE}/failure_cases.zip